In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "dbmdz/bert-base-italian-xxl-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Model loaded on:", device)

In [ ]:
import numpy as np

def get_embedding(text: str) -> np.ndarray:
    """
    Tokenize text, run it through the Italian BERT model, mean-pool the
    token embeddings into a single 768-dim vector per segment.
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():  # not training -- skip gradient computation
        outputs = model(**inputs)

    token_embeddings = outputs.last_hidden_state[0]   # shape: (num_tokens, 768)
    segment_embedding = token_embeddings.mean(dim=0)  # mean pool -> shape: (768,)

    return segment_embedding.cpu().numpy()

In [ ]:
test_file = next((SEGMENTS_DIR / "HC").glob("*_N1_seg1.txt"))
print("Testing on:", test_file.name)

text = test_file.read_text(encoding="utf-8")
print("Text preview:", text[:100], "...")
print("Text word count:", len(text.split()))

embedding = get_embedding(text)
print("Embedding shape:", embedding.shape)
print("Embedding dtype:", embedding.dtype)
print("First 10 values:", embedding[:10])
print("Any NaN?", np.isnan(embedding).any())

In [ ]:
from tqdm import tqdm  # progress bar; already installed with transformers

all_segments = sorted((SEGMENTS_DIR / "HC").glob("*.txt")) + sorted((SEGMENTS_DIR / "PT").glob("*.txt"))
print(f"Found {len(all_segments)} segment files")

done, skipped, failed = 0, 0, 0

for seg_path in tqdm(all_segments):
    label = seg_path.parent.name   # 'HC' or 'PT'
    out_path = FEATURES_DIR / label / f"{seg_path.stem}.npy"

    if out_path.exists():
        skipped += 1
        continue

    try:
        text = seg_path.read_text(encoding="utf-8")
        if not text.strip():
            print(f"  !! EMPTY: {seg_path.name}")
            failed += 1
            continue
        embedding = get_embedding(text)
        np.save(out_path, embedding)
        done += 1
    except Exception as e:
        print(f"  !! FAILED on {seg_path.name}: {e}")
        failed += 1

print(f"\nDone. {done} extracted, {skipped} already done, {failed} failed.")

In [ ]:
segments_hc = set(p.stem for p in (SEGMENTS_DIR / "HC").glob("*.txt"))
segments_pt = set(p.stem for p in (SEGMENTS_DIR / "PT").glob("*.txt"))

features_hc = set(p.stem for p in (FEATURES_DIR / "HC").glob("*.npy"))
features_pt = set(p.stem for p in (FEATURES_DIR / "PT").glob("*.npy"))

print(f"Segments: HC={len(segments_hc)}, PT={len(segments_pt)}")
print(f"Features: HC={len(features_hc)}, PT={len(features_pt)}")

missing_hc = segments_hc - features_hc
missing_pt = segments_pt - features_pt
print(f"Missing HC features: {len(missing_hc)}")
print(f"Missing PT features: {len(missing_pt)}")

In [ ]:
all_features = sorted(FEATURES_DIR.glob("*/*.npy"))
print(f"Checking {len(all_features)} feature files...")

bad_shape = []
has_nan = []
all_zero = []

for f in all_features:
    vec = np.load(f)
    if vec.shape != (768,):
        bad_shape.append(f.name)
    if np.isnan(vec).any():
        has_nan.append(f.name)
    if np.all(vec == 0):
        all_zero.append(f.name)

print(f"Wrong shape: {len(bad_shape)}")
print(f"Contains NaN: {len(has_nan)}")
print(f"All-zero (suspicious): {len(all_zero)}")

In [ ]:
# Two segments from very different speakers should NOT have identical embeddings
sample_hc = np.load(sorted((FEATURES_DIR / "HC").glob("*.npy"))[0])
sample_pt = np.load(sorted((FEATURES_DIR / "PT").glob("*.npy"))[0])

print("Are they identical?", np.array_equal(sample_hc, sample_pt))
print("Max abs difference:", np.abs(sample_hc - sample_pt).max())